# 04b · RAD-DINO LP-FT — ML-Decoder head + Distribution-Balanced Loss

Retrains the CXR backbone from `microsoft/rad-dino` on the **rebuilt cohort**. The
old `raddino_lpft_mv73_dbml_best.pt` is not reused: it was fitted on the cohort we
threw away.

**Tuan's changes to the plan:**

| | PLAN §B | here |
|---|---|---|
| head | linear on CLS | **ML-Decoder** — class queries cross-attending to the *spatial* tokens |
| loss | (fusion only) | **DB Loss** in LP-FT too |
| multi-view | mean-pool frontal | **weighted average of probabilities**, `w_f=0.7 / w_l=0.3` |

**Trained per image, evaluated per study.** The 7:3 average is a *study-level*
combination of per-image probabilities:

```
p_study = ( w_f·mean(p over frontal) + w_l·mean(p over lateral) ) / (w_f·[has frontal] + w_l·[has lateral])
```

That denominator is not decoration. **51.6% of the cohort is frontal-only** (03), so
without renormalising, half the studies would get `0.7 · p` — a systematic 30%
deflation that destroys calibration and every per-label threshold in §C. It is a
weighted *average*, not a weighted *sum*.

**Ports, not re-derivations** (§B8: *"Do not re-derive the constants — port the
official implementation"*). ML-Decoder comes from `Alibaba-MIIL/ML_Decoder`; DB Loss
is `wutong16/DistributionBalancedLoss`'s `ResampleLoss` with the math copied
verbatim and only its I/O adapted.

**Prereq:** `04a` must have finished — this notebook reads the shards, never the
615 GB originals.


## C0 · Config

In [ ]:
import os, io, re, time, json, math, tarfile, shutil, pickle, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm      # real progress bars; Colab renders bare CR-updates badly

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR  = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
CKPT_DIR  = os.path.join(DATA_DIR, 'lpft_v2'); os.makedirs(CKPT_DIR, exist_ok=True)
LOCAL_IMG = '/content/cxr_local'                       # extracted here, NOT Drive

# ---- multi-view weighted average (eval only; training is per-image) ----
W_F, W_L = 0.7, 0.3                                    # ablate these; they must sum to 1
assert abs(W_F + W_L - 1.0) < 1e-9

# ---- schedule ----
LP_EPOCHS, FT_EPOCHS = 4, 12
BATCH, ACCUM         = 48, 1                           # A6000 48GB @518^2 (1370 tokens)
LP_LR, FT_LR         = 1e-3, 1e-5                      # head-only vs whole-backbone
HEAD_WD, BB_WD       = 1e-2, 1e-4
NUM_WORKERS          = 8
SEED                 = 42
AMP_DTYPE            = torch.bfloat16                  # A6000 = Ampere, bf16 is native

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.cuda.get_device_name(0) if dev=='cuda' else 'NO GPU')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB' if dev=='cuda' else '')

ValueError: mount failed

## C1 · Stage the shards on **local disk**

Drive tops out at ~80 MB/s and that is the VM's ceiling, not Drive's (04a measured
GCS at the same number). Training from Drive would idle the A6000 for 2h an epoch.
Copying is sequential over a few hundred ~50 MB files — the access pattern Drive is
actually good at — and then every epoch reads from local SSD.

Copy → extract → delete the local tar, one shard at a time, so peak disk is
`extracted + 1 tar` rather than double.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
print(f'shards on Drive: {len(tars)}   '
      f'({sum(os.path.getsize(os.path.join(SHARD_DIR,f)) for f in tars)/1e9:.1f} GB)')
assert tars, 'no shards -> run 04a first'

print(f'images already local: {len(os.listdir(LOCAL_IMG)):,}')
todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
print(f'shards still to stage: {len(todo)} / {len(tars)}   (~8 min for a full set)')
for f in tqdm(todo, desc='staging shards', unit='shard'):
    tmp = f'/content/{f}'
    shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)          # sequential: Drive is fine here
    with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
    os.remove(tmp)
    open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()  # marker AFTER extract
n_local = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
print(f'\nlocal images: {n_local:,}')
print(f'local disk used: {sum(os.path.getsize(os.path.join(LOCAL_IMG,f)) for f in os.listdir(LOCAL_IMG))/1e9:.1f} GB')
!df -h /content | tail -1

shards on Drive: 347   (16.9 GB)
images already local: 0
shards still to stage: 347 / 347   (~8 min for a full set)


staging shards:   0%|          | 0/347 [00:00<?, ?shard/s]

/tmp/ipykernel_3135/4148172763.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)



local images: 346,578
local disk used: 16.6 GB
overlay         236G   79G  158G  34% /


## C2 · ML-Decoder — fetched from the official repo

§B8's rule applied to the head: **do not re-implement it**. A hand-rolled
transformer-decoder head with a subtly wrong attention mask trains fine and is
simply worse, which is indistinguishable from "the idea did not work".

With 13 classes ML-Decoder degenerates to 13 learned queries (one per label), each
cross-attending over RAD-DINO's **1369 patch tokens**, then a per-query projection
to its own logit. That is the whole point of using it over a linear head on CLS:
each pathology can attend to the lung region where it lives, instead of everything
being squeezed through one pooled 768-d vector.

In [ ]:
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
print('ml_decoder.py bytes:', os.path.getsize(ML_SRC))
# imported here, not relied on from C0: this cell gets re-run on its own whenever the
# shim changes, and a NameError from a stale C0 is a waste of everyone's time
import sys, types, inspect, re, torch
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
print(inspect.signature(MLDecoder.__init__))

# ---- PyTorch 2.x compatibility shims ----------------------------------------
# ML-Decoder's TransformerDecoderLayerOptimal (2021) is driven by torch's
# nn.TransformerDecoder, which has since grown two expectations this layer does not
# meet. BOTH concern SELF-attention -- and this layer deliberately has none: that is
# what "Optimal" means here. The class queries never attend to each other, only to the
# image tokens (cross-attention + FFN only). So both shims are provably no-ops for the
# math; neither adds, removes, or reweights a single operation.
#
#   (1) forward() does:  _get_seq_len(tgt, self.layers[0].self_attn.batch_first)
#       -> AttributeError. It reads ONLY `.batch_first`, so a flag-only stub suffices.
#          Queries are fed as [n_queries, batch, dim] -> seq-first -> False.
#
#   (2) forward() now passes tgt_is_causal / memory_is_causal down to every layer
#       -> TypeError. Both describe a causal mask over self-attention. No
#          self-attention, nothing to mask, safe to drop.
#
# Unknown future kwargs are deliberately NOT swallowed: if torch adds one that does
# change semantics, it should break loudly here rather than train quietly wrong.
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched_for_torch2'):
    _orig_init = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw):
        _orig_init(self, *a, **kw)
        self.self_attn = types.SimpleNamespace(batch_first=False)      # shim (1)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init

    _orig_fwd = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None)                                  # shim (2)
        kw.pop('memory_is_causal', None)
        return _orig_fwd(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd

    _mld.TransformerDecoderLayerOptimal._patched_for_torch2 = True
    print(f'\npatched TransformerDecoderLayerOptimal for torch {torch.__version__}:')
    print('   (1) self_attn stub carrying only .batch_first  -- no self-attention added')
    print('   (2) drop tgt_is_causal / memory_is_causal      -- no self-attn to mask')
    print(f'   layer forward accepts: {sorted(inspect.signature(_orig_fwd).parameters)}')

# prove the layer really has no self-attention, i.e. the stub is a shim not a fix
_src = open(ML_SRC).read()
_cls = _src[_src.index('class TransformerDecoderLayerOptimal'):]
_cls = _cls[:_cls.index('\nclass ') if '\nclass ' in _cls[1:] else len(_cls)]
print('\nlayer defines:', sorted(set(re.findall(r'self\.(\w+)\s*=\s*nn\.', _cls))))
assert 'multihead_attn' in _cls, 'unexpected ML-Decoder source -- inspect it before trusting'

ml_decoder.py bytes: 7934
(self, num_classes, num_of_groups=-1, decoder_embedding=768, initial_num_features=2048, zsl=0)

patched TransformerDecoderLayerOptimal for torch 2.11.0+cu128:
   (1) self_attn stub carrying only .batch_first  -- no self-attention added
   (2) drop tgt_is_causal / memory_is_causal      -- no self-attn to mask
   layer forward accepts: ['memory', 'memory_key_padding_mask', 'memory_mask', 'self', 'tgt', 'tgt_key_padding_mask', 'tgt_mask']

layer defines: ['dropout', 'dropout1', 'dropout2', 'dropout3', 'linear1', 'linear2', 'multihead_attn', 'norm1', 'norm2', 'norm3']


## C3 · Distribution-Balanced Loss — `ResampleLoss`, math verbatim

Ported from `wutong16/DistributionBalancedLoss` (`ResampleLoss`,
`reweight_func='rebalance'`). It cannot be `wget`-ed: the file imports from that
repo's `mmcv`/`mllt` registry. So the class is vendored here with those helpers
inlined and **only the I/O adapted** — every formula is the original.

Two mechanisms, per §B8:
- **rebalanced weighting** — corrects expected-vs-observed sampling frequency under
  label co-occurrence (plain resampling double-counts an image carrying several
  labels)
- **negative-tolerant regularisation (NTR)** — a class-aware bias + `neg_scale` so
  the flood of negatives on a tail class does not drown its rare positive gradients

`class_freq` comes from `02` and **must be in the same label order as the model's
output** — a permuted vector silently applies Pleural Other's weight to Atelectasis.
C3 asserts the order.

In [ ]:
class ResampleLoss(nn.Module):
    """DB Loss (Wu et al., ECCV 2020). Ported from wutong16/DistributionBalancedLoss.
    Formulas verbatim; I/O adapted (no mmcv registry, no .cuda() in __init__)."""
    def __init__(self, class_freq, train_num,
                 use_sigmoid=True, reduction='mean', loss_weight=1.0,
                 reweight_func='rebalance',
                 focal=dict(focal=True, alpha=0.5, gamma=2.0),
                 map_param=dict(alpha=0.1, beta=10.0, gamma=0.9),
                 logit_reg=dict(neg_scale=2.0, init_bias=0.05)):
        super().__init__()
        assert use_sigmoid
        self.reduction, self.loss_weight, self.reweight_func = reduction, loss_weight, reweight_func
        self.focal    = focal['focal']; self.gamma = focal['gamma']; self.alpha = focal['alpha']
        self.map_alpha, self.map_beta, self.map_gamma = \
            map_param['alpha'], map_param['beta'], map_param['gamma']

        cf = torch.as_tensor(np.asarray(class_freq), dtype=torch.float)
        self.register_buffer('class_freq', cf)
        self.train_num   = float(train_num)
        self.register_buffer('freq_inv',      torch.ones_like(cf) / cf)
        self.register_buffer('propotion_inv', self.train_num / cf)

        self.neg_scale = logit_reg.get('neg_scale', 1.0)
        init_bias      = logit_reg.get('init_bias', 0.0)
        # class-aware prior bias (NTR). Original: -log(train_num/class_freq - 1)*init_bias/neg_scale
        self.register_buffer('init_bias',
            -torch.log(self.train_num / cf - 1) * init_bias / self.neg_scale)
        self.logit_reg = logit_reg

    def rebalance_weight(self, gt_labels):
        repeat_rate = torch.sum(gt_labels * self.freq_inv, dim=1, keepdim=True)
        pos_weight  = self.freq_inv.unsqueeze(0) / repeat_rate
        # pos and neg treated equally
        return torch.sigmoid(self.map_beta * (pos_weight - self.map_gamma)) + self.map_alpha

    def logit_reg_functions(self, labels, logits, weight):
        if not self.logit_reg: return logits, weight
        if 'init_bias' in self.logit_reg:
            logits = logits + self.init_bias
        if 'neg_scale' in self.logit_reg:
            logits = logits * (1 - labels) * self.neg_scale + logits * labels
            weight = weight / self.neg_scale * (1 - labels) + weight * labels
        return logits, weight

    def forward(self, cls_score, label):
        label  = label.float()
        weight = self.rebalance_weight(label) if self.reweight_func == 'rebalance' \
                 else torch.ones_like(label)
        cls_score, weight = self.logit_reg_functions(label, cls_score, weight)
        if self.focal:
            logpt = F.binary_cross_entropy_with_logits(cls_score, label, reduction='none')
            pt    = torch.exp(-logpt)
            wtloss  = F.binary_cross_entropy_with_logits(cls_score, label,
                                                         weight=weight, reduction='none')
            alpha_t = torch.where(label == 1, self.alpha, 1 - self.alpha)
            loss = alpha_t * ((1 - pt) ** self.gamma) * wtloss
        else:
            loss = F.binary_cross_entropy_with_logits(cls_score, label,
                                                      weight=weight, reduction='none')
        if self.reduction == 'mean': loss = loss.mean()
        elif self.reduction == 'sum': loss = loss.sum()
        return self.loss_weight * loss

cf = pickle.load(open(os.path.join(OUT_DIR,'class_freq.pkl'),'rb'))
meta   = json.load(open(os.path.join(OUT_DIR,'cohort_meta.json')))
LABELS = meta['labels']
assert cf.get('labels', LABELS) == LABELS, \
    'class_freq label order != cohort_meta order -> DB Loss would weight the wrong class'
assert len(cf['class_freq']) == len(LABELS) == 13
print('class_freq (train positives):')
for l, n in zip(LABELS, cf['class_freq']): print(f'   {l:<28} {int(n):>7,}')
print(f'\nnum_train {cf["num_train"]:,}  |  imbalance '
      f'{max(cf["class_freq"])/min(cf["class_freq"]):.1f}x')

class_freq (train positives):
   Atelectasis                   36,972
   Cardiomegaly                  33,305
   Consolidation                  9,934
   Edema                         26,553
   Enlarged_Cardiomediastinum    10,848
   Fracture                       3,212
   Lung_Lesion                    4,761
   Lung_Opacity                  36,179
   No_Finding                    48,443
   Pleural_Effusion              39,675
   Pleural_Other                  1,763
   Pneumonia                     22,603
   Pneumothorax                   7,555

num_train 147,791  |  imbalance 27.5x


## C4 · Dataset — per image, geometry inherited from 04a

04a already applied RAD-DINO's own resize rule (shortest edge → center crop) at
`STORE_SIZE`, so here we only crop to `MODEL_SIZE` and normalise with the
processor's constants. Stored grayscale is repeated to 3 channels: RAD-DINO's
`image_mean/std` are identical across channels (`0.5307/0.2583`), so it always
expected replicated grayscale — this is exact, not an approximation.

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image

proc       = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']
MEAN, STD  = proc.image_mean[0], proc.image_std[0]
print(f'MODEL_SIZE {MODEL_SIZE} | mean {MEAN} std {STD}')

train_tf = T.Compose([
    T.RandomResizedCrop(MODEL_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                        interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.0),      # OFF: laterality matters on a chest film
    T.RandomRotation(7),
    T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3),
])
eval_tf = T.Compose([
    T.CenterCrop(MODEL_SIZE),
    T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3),
])

base = pd.read_parquet(os.path.join(OUT_DIR,'cohort_base.parquet'))
vm   = pd.read_parquet(os.path.join(OUT_DIR,'view_map.parquet'))
items = (vm[vm.view_group.isin(['frontal','lateral'])]
           .merge(base[['study_id','split'] + LABELS], on='study_id', how='inner'))

# Dropping images whose shard entry is missing must be LOUD. A silent filter here
# would quietly shrink the dataset, and -- worse -- a loss that is SKEWED is far more
# damaging than a loss that is large: all-lateral loss rewrites the 7:3 weighting,
# all-train loss invalidates class_freq. Size alone does not reveal either.
before = len(items)
have   = items.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
lost   = items[~have]
items  = items[have].reset_index(drop=True)
print(f'\nimages: {len(items):,} / {before:,}   dropped {len(lost):,} with no local file')

if len(lost):
    print('\n!! images dropped -- cross-check SHARD_DIR/_errors.csv from 04a')
    allx = pd.concat([items, lost])
    for col in ('view_group', 'split'):
        sl = lost[col].value_counts(normalize=True)
        sa = allx[col].value_counts(normalize=True)
        skew = (sl - sa).abs().max()
        print(f'   {col:<11} loss share {dict(lost[col].value_counts())}   '
              f'max skew vs cohort {skew:.3f}' + ('  <-- SKEWED' if skew > 0.05 else ''))
assert len(lost) / before < 0.01, \
    f'{len(lost):,} images ({len(lost)/before:.2%}) missing -> fix 04a before training'

# The 7:3 weighting was characterised on 03's composition. If losing images moved a
# study out of "both views", the weighting now applies to a different population than
# the one we measured -- so re-derive it here rather than trusting 03's numbers.
g = (items.assign(f=items.view_group.eq('frontal'), l=items.view_group.eq('lateral'))
          .groupby('study_id')[['f','l']].any())
comp = pd.Series(np.select([g.f & g.l, g.f & ~g.l, ~g.f & g.l],
                           ['both','frontal only','lateral only'], 'none'), index=g.index)
print(f'\nstudies: {len(g):,}   view composition AS TRAINED (03 measured: '
      f'both 48.18% / frontal only 51.65% / lateral only 0.17%):')
print(comp.value_counts().to_frame('n_studies')
        .assign(frac=lambda d: (d.n_studies/len(g)).round(4)).to_string())
assert (comp != 'none').all(), 'a study lost every image -> it would vanish from eval'
print(f'\n-> the w_f/w_l sweep only moves the "both" rows; the rest renormalise to 1.0')
print('\nby split:'); print(items.groupby('split').size().to_frame('n_images').to_string())

class CXR(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True); self.tf = tf
        self.y  = df[LABELS].to_numpy('float32')
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r  = self.df.iloc[i]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))          # RAD-DINO wants 3 identical channels
        return self.tf(im), torch.from_numpy(self.y[i]), i

def loader(split, tf, shuffle, bs=BATCH):
    return DataLoader(CXR(items[items.split == split], tf), batch_size=bs, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=shuffle,
                      persistent_workers=NUM_WORKERS > 0)

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

MODEL_SIZE 518 | mean 0.5307 std 0.2583

images: 346,578 / 346,578   dropped 0 with no local file

studies: 211,130   view composition AS TRAINED (03 measured: both 48.18% / frontal only 51.65% / lateral only 0.17%):
              n_studies    frac
frontal only     109052  0.5165
both             101712  0.4818
lateral only        366  0.0017

-> the w_f/w_l sweep only moves the "both" rows; the rest renormalise to 1.0

by split:
       n_images
split          
test      52201
train    242497
val       51880


## C5 · Model — RAD-DINO backbone + ML-Decoder over patch tokens

`AutoModel` returns `last_hidden_state = [B, 1+1369, 768]`: CLS then patches.
**ML-Decoder receives the patch tokens, not CLS** — attending over the spatial grid
is the entire reason for the head.

CLS is still what `05` will extract as the frozen `x_cxr` for FiLM fusion. It is not
wasted: fine-tuning backprops through the backbone, so CLS is shaped by this
training too — it just will not carry ML-Decoder's per-class localisation.

In [ ]:
from transformers import AutoModel

class RadDinoMLD(nn.Module):
    def __init__(self, n_classes=13):
        super().__init__()
        self.backbone = AutoModel.from_pretrained('microsoft/rad-dino')
        d = self.backbone.config.hidden_size
        self.head = MLDecoder(num_classes=n_classes, initial_num_features=d,
                              decoder_embedding=768, num_of_groups=n_classes)
        self.d = d
    def forward(self, x, return_cls=False):
        h   = self.backbone(pixel_values=x).last_hidden_state    # [B, 1+P, d]
        cls, patches = h[:, 0], h[:, 1:]                         # ML-Decoder eats patches
        # MLDecoder expects CNN-style [B, C, H, W]; give it [B, C, P, 1] so its
        # flatten(2).transpose(1,2) yields the [B, P, C] token sequence unchanged.
        logits = self.head(patches.transpose(1, 2).unsqueeze(-1))
        return (logits, cls) if return_cls else logits

model = RadDinoMLD(len(LABELS)).to(dev)
n_bb   = sum(p.numel() for p in model.backbone.parameters())
n_head = sum(p.numel() for p in model.head.parameters())
print(f'backbone {n_bb/1e6:.1f} M | head {n_head/1e6:.1f} M')
with torch.no_grad():
    o = model(torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev))
print('logits', tuple(o.shape))
assert o.shape == (2, len(LABELS)), o.shape

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

backbone 86.6 M | head 6.1 M
logits (2, 13)


## C6 · §E2 gates — before spending 13 GPU-hours

`overfit-one-batch` is the one that matters: if the model cannot drive loss to ~0 on
a single batch, the wiring is broken, and no amount of epochs will reveal it as
clearly as three minutes here.

In [ ]:
crit = ResampleLoss(class_freq=cf['class_freq'], train_num=cf['num_train']).to(dev)

# --- DB loss is finite and class-aware: rarer class must get a larger weight ---
lb = torch.zeros(4, len(LABELS), device=dev); lb[:, 0] = 1
w  = crit.rebalance_weight(lb)
order = np.argsort(cf['class_freq'])                     # rarest first
wm = w.mean(0).detach().cpu().numpy()
print('rebalance weight vs train frequency:')
for i in order[:3]:  print(f'   RARE {LABELS[i]:<28} freq {int(cf["class_freq"][i]):>7,}  w {wm[i]:.4f}')
for i in order[-3:]: print(f'   HEAD {LABELS[i]:<28} freq {int(cf["class_freq"][i]):>7,}  w {wm[i]:.4f}')
assert np.isfinite(wm).all(), 'DB weights not finite'
assert wm[order[0]] > wm[order[-1]], 'rarest class must outweigh the most common one'
print('OK  DB loss is class-aware in the right direction')

# --- overfit one batch ---
model.train()
xb, yb, _ = next(iter(loader('train', train_tf, True, bs=8)))
xb, yb = xb.to(dev), yb.to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
l0 = None
for step in range(60):
    with torch.autocast('cuda', dtype=AMP_DTYPE):
        loss = crit(model(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step == 0: l0 = loss.item()
    if step % 20 == 0: print(f'   step {step:>2}  loss {loss.item():.4f}')
print(f'loss {l0:.4f} -> {loss.item():.4f}')
assert loss.item() < l0 * 0.5, 'cannot overfit one batch -> wiring is broken, do NOT train'
print('OK  overfit-one-batch')
del opt; torch.cuda.empty_cache()

rebalance weight vs train frequency:
   RARE Pleural_Other                freq   1,763  w 1.1000
   RARE Fracture                     freq   3,212  w 1.1000
   RARE Lung_Lesion                  freq   4,761  w 1.1000
   HEAD Atelectasis                  freq  36,972  w 0.8311
   HEAD Pleural_Effusion             freq  39,675  w 0.6790
   HEAD No_Finding                   freq  48,443  w 0.3030
OK  DB loss is class-aware in the right direction
   step  0  loss 0.0921
   step 20  loss 0.0013
   step 40  loss 0.0000
loss 0.0921 -> 0.0000
OK  overfit-one-batch


## C7 · Eval — per-image → study, the 7:3 average **with its denominator**

The renormalisation is the whole correctness story here, so it is written once and
asserted: a frontal-only study must come out exactly equal to its frontal mean, not
0.7× it.

In [ ]:
@torch.no_grad()
def predict(split):
    model.eval()
    dl = loader(split, eval_tf, False, bs=BATCH*2)
    df = items[items.split == split].reset_index(drop=True)
    P  = np.zeros((len(df), len(LABELS)), 'float32')
    for xb, _, idx in tqdm(dl, desc=f'predict[{split}]', unit='batch', leave=False):
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            p = torch.sigmoid(model(xb.to(dev, non_blocking=True))).float()
        P[idx.numpy()] = p.cpu().numpy()
    return df, P

def study_level(df, P, w_f=W_F, w_l=W_L, on_zero='error'):
    """p_study = (w_f*mean_frontal + w_l*mean_lateral) / (w_f*has_f + w_l*has_l)

    The denominator uses only the view groups PRESENT. 51.6% of studies are
    frontal-only; a plain weighted SUM would hand them 0.7*p and deflate every
    probability by 30%.

    on_zero: what to do when a study's only view carries weight 0 -- which happens
    at the sweep endpoints (w_f=0 leaves 51.6% of studies with no usable evidence,
    w_f=1 leaves the 366 lateral-only ones). 'error' for the real config, 'nan' for
    the sweep, where those studies are dropped and COUNTED rather than quietly
    back-filled with the view we just said to ignore."""
    d = pd.concat([df[['study_id','view_group']].reset_index(drop=True),
                   pd.DataFrame(P, columns=LABELS)], axis=1)
    sid = np.sort(d.study_id.unique())
    g   = d.groupby(['study_id','view_group'])[LABELS].mean()

    def side(v):
        # a study missing this view -> NaN row, which then contributes 0 weight
        if v not in g.index.get_level_values('view_group'):
            return np.full((len(sid), len(LABELS)), np.nan, 'float32')
        return g.xs(v, level='view_group').reindex(sid)[LABELS].to_numpy('float32')

    pf, pl = side('frontal'), side('lateral')
    hf, hl = ~np.isnan(pf[:, :1]), ~np.isnan(pl[:, :1])       # which views this study HAS
    num = np.nan_to_num(pf) * w_f * hf + np.nan_to_num(pl) * w_l * hl
    den = w_f * hf + w_l * hl                                  # <- only the views present
    usable = (den > 0).ravel()
    if not usable.all():
        if on_zero == 'error':
            raise AssertionError(
                f'{(~usable).sum()} studies have zero total weight (their only view '
                f'carries w=0 under w_f={w_f}, w_l={w_l}). Use on_zero="nan" if this '
                f'is a sweep endpoint.')
        num, den, sid = num[usable], den[usable], sid[usable]
    return sid, (num / den).astype('float32')

# --- assert the renormalisation on a synthetic case ---
_d = pd.DataFrame({'study_id':[1,1,2],'view_group':['frontal','lateral','frontal']})
_P = np.array([[0.8]*13, [0.4]*13, [0.6]*13], 'float32')
_s, _p = study_level(_d, _P)
assert abs(_p[0,0] - (0.7*0.8 + 0.3*0.4)) < 1e-6, 'both-view study wrong'
assert abs(_p[1,0] - 0.6) < 1e-6, f'frontal-only must equal its own mean, got {_p[1,0]:.4f}'
print('OK  7:3 renormalisation: both-view -> 0.68 ; frontal-only -> 0.60 (not 0.42)')

OK  7:3 renormalisation: both-view -> 0.68 ; frontal-only -> 0.60 (not 0.42)


## C8 · Metrics — §C: mAP primary, AUROC auxiliary, tail broken out

*"AUROC is inflated at <2% prevalence"* — so macro **AUPRC** decides the checkpoint.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

TRAIN_PREV = np.asarray(cf['class_freq'], float) / cf['num_train']
TIER = np.where(TRAIN_PREV < 0.02, 'tail', np.where(TRAIN_PREV < 0.10, 'medium', 'head'))

def metrics(sid, P, split):
    Y = (base.set_index('study_id').loc[sid, LABELS].to_numpy('float32'))
    ap  = np.array([average_precision_score(Y[:,j], P[:,j]) for j in range(len(LABELS))])
    auc = np.array([roc_auc_score(Y[:,j], P[:,j]) if len(np.unique(Y[:,j]))>1 else np.nan
                    for j in range(len(LABELS))])
    r = pd.DataFrame({'label':LABELS,'tier':TIER,'train_prev':TRAIN_PREV.round(4),
                      'n_pos':Y.sum(0).astype(int),'AUPRC':ap.round(4),'AUROC':auc.round(4)})
    print(f'\n=== {split} ===  macro mAP {ap.mean():.4f} | macro AUROC {np.nanmean(auc):.4f}')
    print(r.sort_values('train_prev').to_string(index=False))
    print('\nby tier:'); print(r.groupby('tier')[['AUPRC','AUROC']].mean().round(4).to_string())
    return ap.mean()

## C9 · LP → FT, **resumable**

**LP** trains only ML-Decoder on the frozen backbone: the head starts random, and
letting random gradients into pretrained weights is what erases the representation
LP-FT exists to protect. **FT** then unfreezes at a 100× smaller LR.

Checkpoint on **best val macro AUPRC** (§C), never on loss.

**Resume.** LP 4 + FT 12 epochs ≈ 13h on the A6000; Colab will disconnect several
times. Two checkpoints, different jobs:

| file | written | purpose |
|---|---|---|
| `*_last.pt` | **every** epoch | resume: model + **optimizer + scheduler + epoch** |
| `*_best.pt` | on best val mAP | the model you actually keep |

Saving the model alone is not enough. `OneCycleLR` would restart from step 0 on a
fresh run — the LR would jump back up to its peak mid-training and wreck the
trajectory — and Adam would lose its moments. Both are restored.

Writes are `.tmp` → `os.replace`, so a disconnect *during* a save cannot leave a
half-written checkpoint that the next run would happily load. Same rule that made
04a's shards safe.

In [ ]:
TAG = 'raddino_mld_db'

def ck_path(name, kind): return os.path.join(CKPT_DIR, f'{TAG}_{name}_{kind}.pt')

def save_ck(path, **kw):
    """Atomic: a kill mid-write leaves a .tmp, never a corrupt .pt that loads wrong."""
    tmp = path + '.tmp'
    torch.save(kw, tmp)
    os.replace(tmp, path)

def load_ck(path):
    # weights_only=False: we wrote this file ourselves and it holds optimizer /
    # scheduler state, not just tensors.
    return torch.load(path, map_location=dev, weights_only=False)

def run_phase(name, epochs, lr, freeze_bb, init_from=None):
    for p in model.backbone.parameters(): p.requires_grad = not freeze_bb
    groups = [{'params': [p for p in model.head.parameters()], 'lr': lr, 'weight_decay': HEAD_WD}]
    if not freeze_bb:
        groups.append({'params': [p for n,p in model.backbone.named_parameters()
                                  if p.ndim > 1], 'lr': lr, 'weight_decay': BB_WD})
        groups.append({'params': [p for n,p in model.backbone.named_parameters()
                                  if p.ndim <= 1], 'lr': lr, 'weight_decay': 0.0})  # norms/biases
    opt = torch.optim.AdamW(groups)
    dl  = loader('train', train_tf, True)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g['lr'] for g in groups],
                                              total_steps=epochs*len(dl), pct_start=0.1)
    start_ep, best = 0, -1.0

    if os.path.exists(ck_path(name, 'last')):
        ck = load_ck(ck_path(name, 'last'))
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sch.load_state_dict(ck['sch'])                    # <- LR picks up mid-cycle
        start_ep, best = ck['epoch'] + 1, ck['best']
        print(f'RESUMED {name} at epoch {start_ep}/{epochs}  (best val mAP so far {best:.4f}, '
              f'lr now {sch.get_last_lr()[0]:.2e})')
    elif init_from and os.path.exists(init_from):
        model.load_state_dict(load_ck(init_from)['model'])
        print(f'{name}: starting from {os.path.basename(init_from)} '
              '(LP best, not LP last -- do not carry a worse epoch into FT)')

    if start_ep >= epochs:
        print(f'{name} already complete (best val mAP {best:.4f}) -- skipping')
        return best

    for ep in range(start_ep, epochs):
        model.train(); t0 = time.time(); tot = 0
        # leave=True: one finished bar PER EPOCH stays in the output, so after a
        # reconnect the cell still shows how the earlier epochs went.
        pbar = tqdm(dl, desc=f'{name.upper()} ep {ep+1}/{epochs}', unit='batch', leave=True)
        for it,(xb,yb,_) in enumerate(pbar):
            xb,yb = xb.to(dev,non_blocking=True), yb.to(dev,non_blocking=True)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                loss = crit(model(xb), yb)
            loss.backward(); opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
            tot += loss.item()
            if it % 20 == 0:                     # .item() already synced; postfix is free
                pbar.set_postfix_str(f'loss {tot/(it+1):.4f} | lr {sch.get_last_lr()[0]:.1e} | '
                                     f'{(it+1)*BATCH/(time.time()-t0):.0f} img/s')
        pbar.close()
        print(f'  {name} ep{ep} train loss {tot/max(len(dl),1):.4f} -- scoring val...')
        df,P = predict('val'); sid,PS = study_level(df,P)
        mAP  = metrics(sid, PS, f'{name} ep{ep} VAL')
        if mAP > best:
            best = mAP
            save_ck(ck_path(name,'best'), model=model.state_dict(), labels=LABELS,
                    phase=name, epoch=ep, val_mAP=mAP, w_f=W_F, w_l=W_L)
            print(f'  * new best (val mAP {mAP:.4f}) -> {TAG}_{name}_best.pt')
        # written EVERY epoch, best or not -- this is the resume point
        save_ck(ck_path(name,'last'), model=model.state_dict(), opt=opt.state_dict(),
                sch=sch.state_dict(), epoch=ep, best=best, phase=name, labels=LABELS)
        print(f'  epoch {ep} done in {(time.time()-t0)/60:.1f}m  '
              f'(resume point saved: {TAG}_{name}_last.pt)')
    return best

print('checkpoints present:', sorted(f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')) or 'none')
print('\n=== LP (backbone frozen) ===')
lp = run_phase('lp', LP_EPOCHS, LP_LR, freeze_bb=True)
print(f'\nLP best val mAP {lp:.4f}\n')
print('=== FT (backbone unfrozen) ===')
ft = run_phase('ft', FT_EPOCHS, FT_LR, freeze_bb=False, init_from=ck_path('lp','best'))
print(f'\nFT best val mAP {ft:.4f}')

checkpoints present: none

=== LP (backbone frozen) ===


LP ep 1/4:   0%|          | 0/5052 [00:00<?, ?batch/s]

  lp ep0 train loss 0.0098 -- scoring val...


predict[val]:   0%|          | 0/541 [00:00<?, ?batch/s]


=== lp ep0 VAL ===  macro mAP 0.2876 | macro AUROC 0.7185
                     label   tier  train_prev  n_pos  AUPRC  AUROC
             Pleural_Other   tail      0.0119    377 0.0309 0.7289
                  Fracture medium      0.0217    690 0.0277 0.5738
               Lung_Lesion medium      0.0322   1020 0.0592 0.6421
              Pneumothorax medium      0.0511   1621 0.1703 0.7462
             Consolidation medium      0.0672   2128 0.1335 0.7178
Enlarged_Cardiomediastinum medium      0.0734   2324 0.1179 0.6675
                 Pneumonia   head      0.1529   4843 0.2463 0.6446
                     Edema   head      0.1797   5691 0.4290 0.8050
              Cardiomegaly   head      0.2254   7137 0.3814 0.7273
              Lung_Opacity   head      0.2448   7753 0.3802 0.6888
               Atelectasis   head      0.2502   7923 0.4078 0.7224
          Pleural_Effusion   head      0.2685   8502 0.6466 0.8478
                No_Finding   head      0.3278  10380 0.7087 0.8284

by

...

FT ep 12/12:   0%|          | 0/5052 [00:00<?, ?batch/s]

  ft ep11 train loss 0.0060 -- scoring val...


predict[val]:   0%|          | 0/541 [00:00<?, ?batch/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c9cd63177e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c9cd63177e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


=== ft ep11 VAL ===  macro mAP 0.4164 | macro AUROC 0.8175
                     label   tier  train_prev  n_pos  AUPRC  AUROC
             Pleural_Other   tail      0.0119    377 0.1048 0.8736
                  Fracture medium      0.0217    690 0.0819 0.7311
               Lung_Lesion medium      0.0322   1020 0.1993 0.8079
              Pneumothorax medium      0.0511   1621 0.5512 0.9014
             Consolidation medium      0.0672   2128 0.1922 0.7824
Enlarged_Cardiomediastinum medium      0.0734   2324 0.1446 0.7192
                 Pneumonia   head      0.1529   4843 0.4242 0.7729
                     Edema   head      0.1797   5691 0.6227 0.8813
              Cardiomegaly   head      0.2254   7137 0.5162 0.8041
              Lung_Opacity   head      0.2448   7753 0.4898 0.7735
               Atelectasis   head      0.2502   7923 0.5463 0.8095
          Pleural_Effusion   head      0.2685   8502 0.7639 0.9030
                No_Finding   head      0.3278  10380 0.7755 0.8676

b

In [ ]:
from sklearn.metrics import average_precision_score

# score on the model we actually keep, not the last-epoch weights sitting in the kernel
best_path = ck_path('ft','best') if os.path.exists(ck_path('ft','best')) else ck_path('lp','best')
_ck = load_ck(best_path); model.load_state_dict(_ck['model'])
print(f'probing {os.path.basename(best_path)}  (val mAP at save: {_ck.get("val_mAP")})')

@torch.no_grad()
def predict_df(sub):
    """predict() but over an arbitrary row subset -- loader() only knows whole splits."""
    model.eval()
    dl = DataLoader(CXR(sub, eval_tf), batch_size=BATCH*2, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    P = np.zeros((len(sub), len(LABELS)), 'float32')
    for xb,_,idx in tqdm(dl, desc='probe', unit='batch', leave=False):
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            p = torch.sigmoid(model(xb.to(dev, non_blocking=True))).float()
        P[idx.numpy()] = p.cpu().numpy()                 # CXR yields positional idx -> aligns with sub
    return sub.reset_index(drop=True), P

def ap_per_label(sid, PS):
    Y = base.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    return np.array([average_precision_score(Y[:,j], PS[:,j]) for j in range(len(LABELS))])

# subsample train to val study-count, seeded (no augmentation: eval_tf, so we measure
# what the backbone learned, not what the RandAug pipeline shows it)
rng = np.random.default_rng(SEED)
tr  = items[items.split=='train']
n_val_studies = items[items.split=='val'].study_id.nunique()
keep = rng.choice(tr.study_id.unique(),
                  size=min(n_val_studies, tr.study_id.nunique()), replace=False)
tr_sub = tr[tr.study_id.isin(keep)]
print(f'train subsample: {tr_sub.study_id.nunique():,} studies / {len(tr_sub):,} images '
      f'(val: {n_val_studies:,} studies)')

dtr,Ptr = predict_df(tr_sub);                    s_tr,PS_tr = study_level(dtr,Ptr)
dva,Pva = predict_df(items[items.split=="val"]); s_va,PS_va = study_level(dva,Pva)
ap_tr, ap_va = ap_per_label(s_tr,PS_tr), ap_per_label(s_va,PS_va)

g = pd.DataFrame({'label':LABELS,'tier':TIER,
                  'train_AP':ap_tr.round(4),'val_AP':ap_va.round(4),
                  'gap':(ap_tr-ap_va).round(4)}).sort_values('gap',ascending=False)
print('\n'+g.to_string(index=False))
print('\nby tier:')
print(g.groupby('tier')[['train_AP','val_AP','gap']].mean().round(4).to_string())

mg = ap_tr.mean()-ap_va.mean()
print(f'\nMACRO  train {ap_tr.mean():.4f} | val {ap_va.mean():.4f} | GAP {mg:+.4f}')
verdict = ('not memorising -- FT slide is DB-Loss tail reallocation; 05/06 embeddings safe' if mg < 0.05
           else 'normal for CXR at this scale -- proceed, read the EHR ablation with care' if mg < 0.15
           else 'MEMORISATION -- freeze backbone for 05 or cross-fit; 06 ablation is not trustworthy as-is')
print('VERDICT:', verdict)
tail_gap = g[g.tier=="tail"].gap.mean()
print(f'tail gap {tail_gap:+.4f} -- '
      + ('tail improves/holds on val => reallocation, not memorisation'
         if tail_gap < 0.05 else 'tail gap is large => consistent with memorising the rare classes'))

probing raddino_mld_db_ft_best.pt  (val mAP at save: 0.4201024520247708)
train subsample: 31,670 studies / 51,985 images (val: 31,670 studies)


probe:   0%|          | 0/542 [00:00<?, ?batch/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c9cd63177e0><function _MultiProcessingDataLoaderIter.__del__ at 0x7c9cd63177e0>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():    
if w.is_alive(): 
           ^ ^ ^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3

probe:   0%|          | 0/541 [00:00<?, ?batch/s]


                     label   tier  train_AP  val_AP    gap
              Pneumothorax medium    0.6652  0.5590 0.1062
               Lung_Lesion medium    0.2994  0.2009 0.0985
              Cardiomegaly   head    0.5727  0.5253 0.0473
                No_Finding   head    0.8314  0.7873 0.0440
                  Fracture medium    0.1089  0.0698 0.0391
               Atelectasis   head    0.5915  0.5536 0.0378
                 Pneumonia   head    0.4619  0.4262 0.0357
             Pleural_Other   tail    0.1402  0.1049 0.0354
                     Edema   head    0.6659  0.6318 0.0341
          Pleural_Effusion   head    0.8054  0.7763 0.0291
             Consolidation medium    0.2132  0.1904 0.0228
              Lung_Opacity   head    0.5008  0.4910 0.0098
Enlarged_Cardiomediastinum medium    0.1522  0.1449 0.0073

by tier:
        train_AP  val_AP     gap
tier                            
head      0.6328  0.5988  0.0340
medium    0.2878  0.2330  0.0548
tail      0.1402  0.1049  0.035

## C10 · Test + the `w_f/w_l` sweep

*"We report results with different values for the weights"* — so the 7:3 is a point
on a curve, not a constant. The sweep is nearly free: the per-image probabilities are
computed once and only the aggregation changes.

In [ ]:
best = os.path.join(CKPT_DIR, 'raddino_mld_db_ft_best.pt')
# load_ck() -> weights_only=False. torch 2.6 flipped that default to True, and this
# checkpoint carries a numpy scalar (val_mAP), which safe-mode refuses to unpickle.
model.load_state_dict(load_ck(best)['model']); print('loaded', best)

df, P = predict('test')
sid, PS = study_level(df, P)
metrics(sid, PS, f'TEST (w_f={W_F}, w_l={W_L})')

print('\n=== w_f sweep (per-image probs reused; only the aggregation changes) ===')
N_ALL = df.study_id.nunique()
rows  = []
for wf in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.3, 0.0]:
    s, p = study_level(df, P, w_f=wf, w_l=1-wf, on_zero='nan')
    Y  = base.set_index('study_id').loc[s, LABELS].to_numpy('float32')
    ap = np.mean([average_precision_score(Y[:,j], p[:,j]) for j in range(len(LABELS))])
    rows.append(dict(w_f=wf, w_l=round(1-wf,2), n_studies=len(s),
                     pct_scored=round(100*len(s)/N_ALL,2), macro_mAP=round(ap,4)))
sw = pd.DataFrame(rows)
print(sw.to_string(index=False))

# Only rows sharing a denominator are comparable. The endpoints do not: at w_f=0 a
# frontal-only study has no evidence at all and is dropped, so its mAP is computed on
# a different -- and easier or harder, unknowably -- population.
full = sw[sw.n_studies == N_ALL]
print(f'\ncomparable rows (all {N_ALL:,} studies scored): {sorted(full.w_f.tolist())}')
if len(full):
    print(f'best w_f among those = {full.loc[full.macro_mAP.idxmax(),"w_f"]}   '
          f'(7:3 gives {sw.loc[sw.w_f==0.7,"macro_mAP"].iloc[0]:.4f})')
drop = sw[sw.n_studies < N_ALL]
if len(drop):
    print('\n!! these endpoints score a DIFFERENT population -- do not read them off the')
    print('   same curve as the rest:')
    print(drop.to_string(index=False))
    print('   w_f=1.0 is the frontal-only baseline (drops the 366 lateral-only studies);')
    print('   w_f=0.0 is a lateral-only baseline and drops HALF the cohort, which is why')
    print('   it is a sanity probe rather than a candidate configuration.')
print('\nNOTE: the weighting only moves the 48.2% of studies that have BOTH views;')
print('      the rest renormalise to w=1.0 whatever w_f is. That flattens the curve.')
print('\nNEXT: 05_cxr_embeddings.ipynb  (extract CLS from this backbone, pool 0.7/0.3)')


loaded /content/drive/MyDrive/MIMIC_CXR/planB_out/lpft_v2/raddino_mld_db_ft_best.pt


predict[test]:   0%|          | 0/544 [00:00<?, ?batch/s]


=== TEST (w_f=0.7, w_l=0.3) ===  macro mAP 0.4199 | macro AUROC 0.8154
                     label   tier  train_prev  n_pos  AUPRC  AUROC
             Pleural_Other   tail      0.0119    379 0.0899 0.8496
                  Fracture medium      0.0217    688 0.0847 0.7140
               Lung_Lesion medium      0.0322   1020 0.2476 0.8169
              Pneumothorax medium      0.0511   1615 0.5550 0.9009
             Consolidation medium      0.0672   2128 0.1864 0.7823
Enlarged_Cardiomediastinum medium      0.0734   2325 0.1457 0.7238
                 Pneumonia   head      0.1529   4846 0.4163 0.7707
                     Edema   head      0.1797   5690 0.6309 0.8850
              Cardiomegaly   head      0.2254   7137 0.5108 0.7994
              Lung_Opacity   head      0.2448   7752 0.4664 0.7596
               Atelectasis   head      0.2502   7923 0.5613 0.8158
          Pleural_Effusion   head      0.2685   8502 0.7818 0.9105
                No_Finding   head      0.3278  10381 0.78